# Experiment 1: Learning Symmetry Detection

**Objective:**
To verify that our implementation of the back-propagation algorithm can learn to solve the "Symmetry Detection" task described in Rumelhart, Hinton, & Williams (1986).

**The Task:**
The network must detect if a 6-bit binary input vector is symmetrical (e.g., `110011`) or asymmetrical (e.g., `110010`).

**Hypothesis:**
By plotting the total error over time, we expect to see a learning curve that starts high and eventually drops to near zero, indicating the network has discovered the underlying rule.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import model

2025-12-24 09:36:37.916312: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-24 09:36:37.947023: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-24 09:36:39.395859: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/saihk/miniconda3/envs/MLshit/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
202

## Experimental Setup

We will use the architecture specified in the paper:
* **Input Layer:** 6 Units (for a 6-bit pattern)
* **Hidden Layer:** 2 Units
* **Output Layer:** 1 Unit

**Hyperparameters:**
* **Learning Rate ($\epsilon$):** 0.1
* **Momentum ($\alpha$):** 0.9
* **Epochs:** 2500

We will track two metrics at every epoch:
1. **Total Error:** The sum of squared errors across all patterns.
2. **Accuracy:** The percentage of patterns correctly classified (Threshold = 0.5).

In [2]:
print("Loading MNIST data...")
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

X_train = X_train_raw.reshape(X_train_raw.shape[0], 784)
X_test = X_test_raw.reshape(X_test_raw.shape[0], 784)

X_train = X_train.astype('float64')
X_test = X_test.astype('float64')

y_train = to_categorical(y_train_raw, 10)
y_test = to_categorical(y_test_raw, 10)

X_train = [x.reshape(1, 784) for x in X_train]
y_train = [y.reshape(1, 10) for y in y_train]

subset_size = 1000
X_train = X_train[:subset_size]
y_train = y_train[:subset_size]

print(f"Data prepared. Training on {len(X_train)} samples.")
print(f"Input shape: {X_train[0].shape}")
print(f"Target shape: {y_train[0].shape}")

Loading MNIST data...
Data prepared. Training on 1000 samples.
Input shape: (1, 784)
Target shape: (1, 10)


## 2. Visualization of Learning

The following plots show the training progress.

* **Left Plot (Error):** Shows the Loss Landscape, which corresponds to the network finding a gradient path towards a solution.
* **Right Plot(Accuracy):** Shows functional performance.

In [5]:
learning_rate = 0.1
momentum = 0.9
epochs = 50

layer_sizes = [784, 30, 10]

weights, biases = model.initialize_network(layer_sizes)
prev_weight_updates = [np.zeros_like(w) for w in weight]
prev_bias_updates = [np.zeros_like(b) for b in biases]

error_history = []
accuracy_history = []

print(f"Starting MNIST training (Architecture: {layer_sizes})..")
for epoch in range(epochs):
    total_error = 0
    correct_count = 0

    for x_case, y_case in zip(X_train, y_train):
        #For forward propagation
        final_output, cache = model.forward_pass(x_case, weights, biases)

        #Error(MSE)
        total_error+= model.calculate_error(final_output, y_case)

        #Accuracy check (Argmax)
        # We find which index has the highest activation (e.g. Index 5)
        prediction = np.argmax(final_output)
        actual = np.argmax(y_case)

        if prediction == actual:
            correct_count+=1

        # Backward
        weight_grads, bias_grads = model.backward_pass(final_output, y_case, cache, weights)

        # Update
        (weights, biases,
         prev_weight_updates,
         prev_bias_updates) = model.update_weights(weights, biases,
                                             weight_grads, bias_grads,
                                             prev_weight_updates, prev_bias_updates,
                                             learning_rate, momentum)

    # Store stats
    avg_error = total_error / len(X_train)
    acc = correct_count / len(X_train) * 100
    error_history.append(avg_error)
    accuracy_history.append(acc)

    print(f"Epoch {epoch + 1}: Error {avg_error:.4f}, Acc {acc:.2f}%")

print(f"Final Training Accuracy: {accuracy_history[-1]}%")

Starting MNIST training (Architecture: [784, 30, 10])..
Epoch 1: Error 0.4627, Acc 14.10%
Epoch 2: Error 0.4601, Acc 13.00%
Epoch 3: Error 0.4684, Acc 11.60%
Epoch 4: Error 0.4682, Acc 9.90%
Epoch 5: Error 0.4682, Acc 10.20%
Epoch 6: Error 0.4679, Acc 10.60%
Epoch 7: Error 0.4674, Acc 10.60%
Epoch 8: Error 0.4666, Acc 10.50%
Epoch 9: Error 0.4660, Acc 10.70%
Epoch 10: Error 0.4655, Acc 10.60%
Epoch 11: Error 0.4652, Acc 10.40%
Epoch 12: Error 0.4649, Acc 10.30%
Epoch 13: Error 0.4647, Acc 10.50%
Epoch 14: Error 0.4645, Acc 10.50%
Epoch 15: Error 0.4644, Acc 10.30%
Epoch 16: Error 0.4642, Acc 10.30%
Epoch 17: Error 0.4641, Acc 10.20%
Epoch 18: Error 0.4640, Acc 10.00%
Epoch 19: Error 0.4639, Acc 10.20%
Epoch 20: Error 0.4638, Acc 10.30%
Epoch 21: Error 0.4637, Acc 10.30%
Epoch 22: Error 0.4636, Acc 10.50%
Epoch 23: Error 0.4635, Acc 10.70%
Epoch 24: Error 0.4634, Acc 10.90%
Epoch 25: Error 0.4633, Acc 11.00%
Epoch 26: Error 0.4632, Acc 11.10%
Epoch 27: Error 0.4631, Acc 11.30%
Epoch 28: